# Vocabulary Fix and Clean Re-encoding - E3

The EDA in notebook (eda_encoding) revealed a vocabulary anomaly. The raw vocabulary contained 1,048,575 distinct entries, which is impossible for real x86 execution. This notebook investigates the root cause, fixes the vocabulary using an opcode whitelist, and re-encodes all 260 trace files with the clean vocabulary in `int16`.

The raw trace files are never modified, will kept as it is. The fix is applied during the encoding pass, so the original data stays untouched and the real execution sequence is preserved.

## Setup the imports and paths

In [1]:
import sys
import os
import time
import re
import numpy as np
import pandas as pd

sys.path.append("..")

from utils import (
    encode_trace_to_npy,
    save_vocab,
    load_vocab,
    check_vocab_size,
    decode_trace_npy,
    load_encoded_trace,
    stream_opcodes,
)
from x86_opcode_whitelist import X86_OPCODE_WHITELIST

In [2]:
RAW_TRACES_DIR = "../sandbox/data-e3/raw_traces"
CLEAN_VOCAB_JSON = "../sandbox/data/vocab_clean.json" # E1 vocab - intentionally not rebuilt for E2 using the same fixed vocab
ENCODED_DIR = "../sandbox/data-e3/encoded_traces"

clean_vocab = load_vocab(CLEAN_VOCAB_JSON)
print(f"Fixed clean vocab loaded: {len(clean_vocab)} opcodes")

Fixed clean vocab loaded: 287 opcodes


----
## Clean Re-encoding with int16

Each raw trace file is streamed once and encoded with the clean vocabulary. Opcodes not in the whitelist are skipped during the pass, they were never real instructions, so removing them does not alter the program's actual execution sequence. The raw `.txt` files are not modified, the original data is preserved, and the clean encoded `.npy` files are written to a new directory.

In [3]:
# The old int32 files are deleted at this point - the entire directory is removed, creating new one
os.makedirs(ENCODED_DIR, exist_ok=True)
print("Encoded directory created")

Encoded directory created


In [4]:
trace_files = sorted([
    os.path.join(RAW_TRACES_DIR, f)
    for f in os.listdir(RAW_TRACES_DIR)
    if f.endswith(".txt")
])

encode_times = {}
encode_counts = {}
skipped = 0

vocab_to_id = {op: i for i, op in enumerate(clean_vocab)}
valid_set = set(clean_vocab)

print(f"Re-encoding {len(trace_files)} files with clean vocab JSON (int16)\n")
total_start = time.time()

# Re-encode all traces with the new clean vocab, saving as int16 numpy arrays
for i, path in enumerate(trace_files, 1):
    filename = os.path.basename(path)
    npy_name = filename.replace(".txt", ".npy")
    npy_path = os.path.join(ENCODED_DIR, npy_name)

    # Skip if already encoded as int16 from a previous run of this notebook
    if os.path.exists(npy_path):
        arr = load_encoded_trace(npy_path)
        if arr.dtype == np.int16:
            encode_counts[filename] = len(arr)
            skipped += 1
            continue
        os.remove(npy_path) # Remove the old int32 file and re-encode as int16 - safty check

    t0 = time.time()
    n = encode_trace_to_npy(path, vocab_to_id, npy_path, valid_opcodes=valid_set)
    elapsed = time.time() - t0

    encode_times[filename] = elapsed
    encode_counts[filename] = n

    if i % 10 == 0 or i == len(trace_files):
        done = time.time() - total_start
        print(f"  [{i:>3}/{len(trace_files)}]  elapsed: {done/60:.1f} min")

print(f"\nRe-encoding complete in {(time.time()-total_start)/60:.1f} min")

if skipped > 0:
    print(f"Skipped {skipped} files already encoded as int16")

Re-encoding 100 files with clean vocab JSON (int16)

  [ 10/100]  elapsed: 10.6 min
  [ 20/100]  elapsed: 19.8 min
  [ 30/100]  elapsed: 28.8 min
  [ 40/100]  elapsed: 39.3 min
  [ 50/100]  elapsed: 47.8 min
  [ 60/100]  elapsed: 56.0 min
  [ 70/100]  elapsed: 66.4 min
  [ 80/100]  elapsed: 76.1 min
  [ 90/100]  elapsed: 84.7 min
  [100/100]  elapsed: 93.8 min

Re-encoding complete in 93.8 min


### Verify the clean encoding

Three checks using a one sample file

1. The `.npy` dtype is `int16`
2. Round trip - decoded opcodes match the raw file with corrupted lines removed, in the same order
3. The dropped count matches the corruption rate

In [5]:
sample_txt = trace_files[0]
sample_name = os.path.basename(sample_txt)
sample_npy = os.path.join(ENCODED_DIR, sample_name.replace(".txt", ".npy"))

arr = load_encoded_trace(sample_npy)
print(f"Sample File: {sample_name}")
print(f"dtype: {arr.dtype}")
print(f"Encoded: {len(arr):,} opcodes")

# Round trip check - first N raw opcodes with corruption removed should equal decoded
N = 100_000
raw_clean = []

for op in stream_opcodes(sample_txt):
    if op in valid_set:
        raw_clean.append(op)
    if len(raw_clean) == N:
        break

decoded = decode_trace_npy(sample_npy, clean_vocab)[:N]

match = raw_clean == decoded
print(f"Round trip check on '{sample_name}' (First {N} opcodes): {'PASS' if match else 'FAIL'}")

# Dropped rate for this file
raw_total = sum(1 for _ in stream_opcodes(sample_txt))
dropped = raw_total - len(arr)

print(f"\nRaw Lines: {raw_total:,}")
print(f"Encoded: {len(arr):,}")
print(f"Dropped: {dropped:,} ({dropped/raw_total*100:.4f}%) - Corrupted lines removed")

Sample File: abbot-0.13.0.txt
dtype: int16
Encoded: 248,811,208 opcodes
Round trip check on 'abbot-0.13.0.txt' (First 100000 opcodes): PASS

Raw Lines: 249,459,514
Encoded: 248,811,208
Dropped: 648,306 (0.2599%) - Corrupted lines removed


### Storage Comparison

The `int16` encoding now showing the full storage reduction that `int32` could not deliver in the previous notebook.

In [6]:
size_rows = []

# Compare the size of the raw text files to the encoded numpy arrays
for path in trace_files:
    filename = os.path.basename(path)
    npy_path = os.path.join(ENCODED_DIR, filename.replace(".txt", ".npy"))
    txt_mb = os.path.getsize(path) / 1e6
    npy_mb = os.path.getsize(npy_path) / 1e6 if os.path.exists(npy_path) else None
    size_rows.append({"filename": filename, "txt_mb": txt_mb, "npy_mb": npy_mb})

df_sizes = pd.DataFrame(size_rows)
df_sizes["reduction"] = 1 - df_sizes["npy_mb"] / df_sizes["txt_mb"]

print("File Size Summary")
print(df_sizes[["txt_mb", "npy_mb", "reduction"]].describe().round(2))
print(f"\nTotal Raw Text: {df_sizes.txt_mb.sum()/1024:.1f} GB")
print(f"Total Encoded: {df_sizes.npy_mb.sum()/1024:.1f} GB")
print(f"Average Reduction: {df_sizes.reduction.mean()*100:.1f}%")

File Size Summary
        txt_mb  npy_mb  reduction
count   100.00  100.00     100.00
mean    966.62  435.04       0.55
std     207.48   93.47       0.00
min     587.35  264.15       0.55
25%     769.59  346.22       0.55
50%    1099.71  495.16       0.55
75%    1144.03  514.23       0.55
max    1188.29  534.83       0.56

Total Raw Text: 94.4 GB
Total Encoded: 42.5 GB
Average Reduction: 55.0%


----

### Method Comparison - Streaming vs Clean Encoded Array

In [7]:
# The streaming timings were saved in eda_encoding notebook - streaming_timing.csv
# It can be compared with this notebook's new clean encoded time data
df_stream = pd.read_csv("../sandbox/data-e3/processed_outputs/streaming_timing.csv")
stream_times_saved = dict(zip(df_stream["filename"], df_stream["stream_sec"]))

# Build a comparison table for files where it has both timings
common = set(stream_times_saved.keys()) & set(encode_times.keys())

if common:
    rows = []

    # Compare the streaming and encoding rates for files that have both timings
    for fname in sorted(common):
        n_ops = encode_counts.get(fname, 0) # Get the number of opcodes
        s_time = stream_times_saved[fname] # Get the streaming times
        e_time = encode_times[fname] # Get the encoding time

        rows.append({
            "filename": fname,
            "opcodes": n_ops,
            "stream_sec":  round(s_time, 2),
            "encode_sec":  round(e_time, 2),
            "stream_rate": int(n_ops / s_time) if s_time > 0 else 0,
            "encode_rate": int(n_ops / e_time) if e_time > 0 else 0,
        })

    df_compare = pd.DataFrame(rows)

    print("Method Comparison - Clean int16 Encoding (Files processed in both passes):\n")
    print(f"  Streaming - AVG Rate: {df_compare['stream_rate'].mean():>10,.0f} opcodes/sec")
    print(f"  Encoding - AVG Rate: {df_compare['encode_rate'].mean():>10,.0f} opcodes/sec")
    print(f"  Encoding is around {df_compare['encode_rate'].mean()/df_compare['stream_rate'].mean():.1f}x faster than streaming")

    df_compare.to_csv("../sandbox/data-e3/processed_outputs/method_comparison_clean.csv", index=False)
    print("\nSaved to method_comparison_clean.csv")

else:
    print("No common files between the saved stream timings and the clean encode timings.")
    print("If all files were already encoded (or skipped), rerun on a new ENCODED_DIR to get encode timings.")

Method Comparison - Clean int16 Encoding (Files processed in both passes):

  Streaming - AVG Rate:  2,420,960 opcodes/sec
  Encoding - AVG Rate:  3,882,967 opcodes/sec
  Encoding is around 1.6x faster than streaming

Saved to method_comparison_clean.csv
